# Lesson 03 — Exercise Solutions

Worked solutions with a short rationale after each. Run setup.sql first.

In [ ]:
USE AdventureWorks2022;
GO

## Exercise 1

Format the ModifiedDate of every Person.Person row as 'DD/MM/YYYY'.
Expected columns: BusinessEntityID, ModifiedDateFormatted
Hint: CONVERT style 103 = DD/MM/YYYY

In [ ]:
SELECT
    BusinessEntityID,
    CONVERT(VARCHAR(10), ModifiedDate, 103) AS ModifiedDateFormatted
FROM Person.Person;

**Why:** CONVERT style 103 produces DD/MM/YYYY directly.

## Exercise 2

Calculate each product's ListPrice rounded to the nearest dollar,
and the difference between the rounded and original price.
Expected columns: Name, ListPrice, Rounded, Difference

In [ ]:
SELECT
    Name,
    ListPrice,
    ROUND(ListPrice, 0)            AS Rounded,
    ListPrice - ROUND(ListPrice, 0) AS Difference
FROM Production.Product
WHERE ListPrice > 0;

**Why:** ROUND(x, 0) rounds to nearest integer; subtraction gives the delta.

## Exercise 3

Some rows in Production.Product have a NULL Weight.
Return all products showing Weight; replace NULL with 0.
Expected columns: Name, Weight, WeightOrZero

In [ ]:
SELECT
    Name,
    Weight,
    ISNULL(Weight, 0) AS WeightOrZero
FROM Production.Product;

**Why:** ISNULL replaces NULL with the specified value; COALESCE is the ANSI equivalent.

## Exercise 4

Convert the SellStartDate of each product to a DATE (stripping time).
Show only products whose SellStartDate year is 2003.
Expected columns: Name, SellStartDate, SellStartDateOnly

In [ ]:
SELECT
    Name,
    SellStartDate,
    CAST(SellStartDate AS DATE) AS SellStartDateOnly
FROM Production.Product
WHERE YEAR(SellStartDate) = 2003;

**Why:** CAST to DATE strips the time component; YEAR() extracts the year for filtering.

## Exercise 5

Using TRY_CAST, attempt to cast the ProductNumber column to INT.
Show ProductNumber, the attempted cast result, and a derived column
IsNumeric (1 if the cast succeeded, 0 if it returned NULL).
Expected columns: ProductNumber, CastResult, IsNumeric

In [ ]:
SELECT
    ProductNumber,
    TRY_CAST(ProductNumber AS INT)                              AS CastResult,
    CASE WHEN TRY_CAST(ProductNumber AS INT) IS NOT NULL
         THEN 1 ELSE 0 END                                      AS IsNumeric
FROM Production.Product;

**Why:** TRY_CAST returns NULL on failure; CASE on the result gives IsNumeric.

## Challenges

Harder problems. They may combine this lesson's topic with
earlier lessons, and usually need 2+ concepts in one query.

### Challenge 1

Demonstrate the MONEY rounding trap: split $100.00 into 3 equal
parts with MONEY and with DECIMAL(19,4), sum the parts back, and show the
difference in cents in one result row.
Expected columns: MoneyTotal, DecimalTotal, MoneyLostCents

In [ ]:
SELECT
    (CAST(100.00 AS MONEY) / 3) * 3                                       AS MoneyTotal,
    CAST((CAST(100.00 AS DECIMAL(19,4)) / 3) * 3 AS DECIMAL(19,4))        AS DecimalTotal,
    (100.00 - CAST((CAST(100.00 AS MONEY) / 3) * 3 AS DECIMAL(19,4))) * 100 AS MoneyLostCents;

**Why:** Divide-then-multiply as one bare expression per type, casting to the final scale only at the end — T-SQL widens a division's intermediate scale, so DECIMAL keeps the extra digits through the multiply and lands on $100.0000. Tempting-but-wrong: declaring the DECIMAL third into a DECIMAL(19,4) variable first — that assignment truncates to 4 places immediately, matching MONEY's $99.9999 and hiding the very trap this challenge demonstrates.

### Challenge 2

Group all 2013 orders into fiscal quarters where the fiscal
year starts July 1 (Q1 = Jul–Sep). Compute the fiscal quarter from
OrderDate with date functions — no CASE ladder over month numbers.
Expected columns: FiscalYear, FiscalQuarter, OrderCount, TotalDue
Hint: DATEADD the date back 6 months, then DATEPART quarter/year of the
shifted date.

In [ ]:
SELECT
    YEAR(DATEADD(MONTH, -6, OrderDate))                AS FiscalYear,
    DATEPART(QUARTER, DATEADD(MONTH, -6, OrderDate))   AS FiscalQuarter,
    COUNT(*)                                            AS OrderCount,
    SUM(TotalDue)                                       AS TotalDue
FROM Sales.SalesOrderHeader
WHERE YEAR(OrderDate) = 2013
GROUP BY YEAR(DATEADD(MONTH, -6, OrderDate)), DATEPART(QUARTER, DATEADD(MONTH, -6, OrderDate))
ORDER BY FiscalYear, FiscalQuarter;

**Why:** Shift OrderDate back 6 months so July lands on "month 1" of the shifted calendar; DATEPART(QUARTER,...) and YEAR(...) of that SAME shifted date give FiscalQuarter and FiscalYear directly, no per-month CASE needed. Tempting-but-wrong: CASE WHEN MONTH(OrderDate) IN (7,8,9) THEN 1... — exactly the ladder the exercise forbids, and it doesn't generalize to other fiscal shapes. 2013 calendar orders span two fiscal years (FY2012 Q3/Q4, FY2013 Q1/Q2).

### Challenge 3

Using Person.Person, produce a padded display name exactly
40 chars wide: "LastName, FirstName" left-aligned, padded with dots. Show
LEN vs DATALENGTH for 5 rows and explain the factor of 2 in a comment.
Expected columns: DisplayName, LenChars, DataLengthBytes
Hint: NVARCHAR: DATALENGTH = 2× LEN.

In [ ]:
SELECT TOP (5)
    LEFT(LastName + ', ' + FirstName + REPLICATE('.', 40), 40)            AS DisplayName,
    LEN(LEFT(LastName + ', ' + FirstName + REPLICATE('.', 40), 40))        AS LenChars,
    DATALENGTH(LEFT(LastName + ', ' + FirstName + REPLICATE('.', 40), 40)) AS DataLengthBytes
FROM Person.Person
ORDER BY BusinessEntityID;

**Why:** Append 40 dots then LEFT-truncate to 40 — guarantees a fixed width regardless of name length. Dots (not spaces) matter: LEN() strips trailing spaces but not dots, so a space-padded string would silently misreport LenChars. NVARCHAR stores 2 bytes/char, so once every row is padded to exactly 40 chars, DataLengthBytes is exactly 2x LenChars (80) on every row.

### Challenge 4

lesson03.MixedDates (seeded in setup.sql) holds date strings
in mixed formats ('2014-03-01', '03/15/2014', '20140401', 'not a date').
Produce a clean report: original string, parsed DATE where possible
(trying styles 120, 101, 112 in that order), and a flag for unparseable
rows.
Expected columns: RawValue, ParsedDate, IsValid
Hint: COALESCE over TRY_CONVERT calls with explicit styles.

In [ ]:
SELECT
    RawValue,
    COALESCE(TRY_CONVERT(DATE, RawValue, 120),
             TRY_CONVERT(DATE, RawValue, 101),
             TRY_CONVERT(DATE, RawValue, 112))              AS ParsedDate,
    CASE WHEN COALESCE(TRY_CONVERT(DATE, RawValue, 120),
                        TRY_CONVERT(DATE, RawValue, 101),
                        TRY_CONVERT(DATE, RawValue, 112)) IS NULL
         THEN 0 ELSE 1 END                                  AS IsValid
FROM lesson03.MixedDates
ORDER BY ID;

**Why:** Try each candidate style with TRY_CONVERT, which returns NULL on a non-matching format instead of erroring; COALESCE picks the first non-NULL hit in priority order (120 ISO, 101 US slash, 112 compact). IsValid flags rows where every style failed. Tempting-but-wrong: a single CAST/CONVERT with one fixed style — it errors on the first mismatched format and kills the whole batch, the exact pitfall this lesson's date-string demo shows.